In [1]:
# 设置工具
from langchain_core.tools import tool
from langgraph.graph import MessagesState, START
from langgraph.prebuilt import ToolNode
from langgraph.graph import END, StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langchain_deepseek import ChatDeepSeek
import os
from dotenv import load_dotenv

load_dotenv()

@tool
def play_song_on_qq(song: str):
    """在qq音乐上播放歌曲"""
    # 调用QQ音乐 API...
    return f"成功在QQ音乐上播放了{song}！"


@tool
def play_song_on_163(song: str):
    """在网易云上播放歌曲"""
    # 调用网易云 API...
    return f"成功在网易云上播放了{song}！"


tools = [play_song_on_qq, play_song_on_163]
tool_node = ToolNode(tools)


deepseek = ChatDeepSeek(
    model="deepseek-v4-flash",
    temperature=0,
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url=os.environ.get("DEEPSEEK_API_BASE"),
)


model = deepseek.bind_tools(tools, parallel_tool_calls=False)


# 定义节点和条件边


# 定义确定是否继续的函数
def should_continue(state):
    messages = state["messages"]
    last_message = messages[-1]
    # 如果没有函数调用，则结束
    if not last_message.tool_calls:
        return "end"
    # 否则如果有，我们继续
    else:
        return "continue"


# 定义调用模型的函数
def call_model(state):
    messages = state["messages"]
    response = model.invoke(messages)
    # 我们返回一个列表，因为这将被添加到现有列表中
    return {"messages": [response]}


# 定义一个新图
workflow = StateGraph(MessagesState)

# 定义我们将循环的两个节点
workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)

# 将入口点设置为`agent`
# 这意味着这个节点是第一个被调用的
workflow.add_edge(START, "agent")

# 现在添加一个条件边
workflow.add_conditional_edges(
    # 首先，我们定义起始节点。我们使用`agent`。
    # 这意味着这些是在调用`agent`节点后采取的边。
    "agent",
    # 接下来，我们传入将确定下一个调用哪个节点的函数。
    should_continue,
    # 最后我们传入一个映射。
    # 键是字符串，值是其他节点。
    # END是一个特殊节点，标记图应该结束。
    # 将会发生的是我们调用`should_continue`，然后该函数的输出
    # 将与此映射中的键匹配。
    # 根据匹配的键，然后调用相应的节点。
    {
        # 如果是`tools`，则调用工具节点。
        "continue": "action",
        # 否则我们结束。
        "end": END,
    },
)

# 现在我们从`tools`到`agent`添加一个普通边。
# 这意味着在调用`tools`之后，下一步调用`agent`节点。
workflow.add_edge("action", "agent")

# 设置内存
memory = MemorySaver()

# 最后，我们编译它！
# 这将它编译成一个LangChain Runnable，
# 意味着你可以像使用任何其他runnable一样使用它

# 我们添加`interrupt_before=["action"]`
# 这将在调用`action`节点之前添加一个断点
app = workflow.compile(checkpointer=memory)

In [2]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}
input_message = HumanMessage(content="你能播放一首周杰伦播放量最高的歌曲吗?")
for event in app.stream({"messages": [input_message]}, config, stream_mode="values"):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

你能播放一首周杰伦播放量最高的歌曲吗?
================================== Ai Message ==================================

好的！周杰伦播放量最高的歌曲，很可能是《七里香》，这首歌在华语乐坛拥有极高的播放量和传唱度。我先在QQ音乐上为您播放这首经典歌曲！
Tool Calls:
  play_song_on_qq (call_00_uqOccBiFxlOTNXJbAqXh9419)
 Call ID: call_00_uqOccBiFxlOTNXJbAqXh9419
  Args:
    song: 七里香
================================= Tool Message =================================
Name: play_song_on_qq

成功在QQ音乐上播放了七里香！
================================== Ai Message ==================================

已在 **QQ音乐** 上成功为您播放了周杰伦的 **《七里香》**！🎵

这首歌不仅是周杰伦的经典代表作之一，也是各大平台播放量名列前茅的神曲。当然，如果您想听其他的热门金曲（比如《晴天》《告白气球》《夜曲》等），我也可以随时为您切换到网易云或其他平台播放哦！😊


In [3]:
app.get_state(config).values["messages"]

[HumanMessage(content='你能播放一首周杰伦播放量最高的歌曲吗?', additional_kwargs={}, response_metadata={}, id='db1ee6dd-764d-4515-8808-c3efcce59646'),
 AIMessage(content='好的！周杰伦播放量最高的歌曲，很可能是《七里香》，这首歌在华语乐坛拥有极高的播放量和传唱度。我先在QQ音乐上为您播放这首经典歌曲！', additional_kwargs={'refusal': None, 'reasoning_content': '用户想播放周杰伦播放量最高的歌曲。周杰伦播放量最高的歌曲通常是《七里香》或者《夜曲》等。不过，我需要确定哪首是播放量最高的。\n\n实际上，周杰伦播放量最高的歌曲可能是《七里香》、《晴天》、《告白气球》等。在QQ音乐上，《七里香》的播放量一直很高，但我不能完全确定哪首是播放量最高的。\n\n我可以先尝试播放《七里香》，这是周杰伦非常经典且播放量很高的歌曲。\n\n或者，我可以播放《晴天》？嗯，让我想想。\n\n在网易云音乐上，《晴天》的评论数最多，但播放量最高的可能是《七里香》或《告白气球》。\n\n不过我可以先尝试播放《七里香》看看。\n\n让我先在QQ音乐上播放《七里香》试试。'}, response_metadata={'token_usage': {'completion_tokens': 267, 'prompt_tokens': 339, 'total_tokens': 606, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 177, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 339}, 'model_provider': 'deepseek', 'model

In [4]:
all_states = []
for state in app.get_state_history(config):
    print(state)
    all_states.append(state)
    print("--")

StateSnapshot(values={'messages': [HumanMessage(content='你能播放一首周杰伦播放量最高的歌曲吗?', additional_kwargs={}, response_metadata={}, id='db1ee6dd-764d-4515-8808-c3efcce59646'), AIMessage(content='好的！周杰伦播放量最高的歌曲，很可能是《七里香》，这首歌在华语乐坛拥有极高的播放量和传唱度。我先在QQ音乐上为您播放这首经典歌曲！', additional_kwargs={'refusal': None, 'reasoning_content': '用户想播放周杰伦播放量最高的歌曲。周杰伦播放量最高的歌曲通常是《七里香》或者《夜曲》等。不过，我需要确定哪首是播放量最高的。\n\n实际上，周杰伦播放量最高的歌曲可能是《七里香》、《晴天》、《告白气球》等。在QQ音乐上，《七里香》的播放量一直很高，但我不能完全确定哪首是播放量最高的。\n\n我可以先尝试播放《七里香》，这是周杰伦非常经典且播放量很高的歌曲。\n\n或者，我可以播放《晴天》？嗯，让我想想。\n\n在网易云音乐上，《晴天》的评论数最多，但播放量最高的可能是《七里香》或《告白气球》。\n\n不过我可以先尝试播放《七里香》看看。\n\n让我先在QQ音乐上播放《七里香》试试。'}, response_metadata={'token_usage': {'completion_tokens': 267, 'prompt_tokens': 339, 'total_tokens': 606, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 177, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 339}, 'mo